In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoder, TransformerDecoderLayer
import re
from tqdm import tqdm
import numpy as np
from matplotlib import pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer

c:\Repositories\proai\course9-genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_ds = load_dataset("parquet", data_files="data/train.parquet")['train']
test_ds = load_dataset("parquet", data_files="data/test.parquet")['train']

In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
NUM_INSTANCES = 10000
MAX_SENT_LEN = 15

inputs, output_dialog = [], []

for i in tqdm(range(NUM_INSTANCES)):

  inputs_sent, dialog_sent = ["<sos>"], ["<sos>"]

  inputs_sent += train_ds[i]['dialog'][0]
  dialog_sent += train_ds[i]['dialog'][1:]

  # change to lowercase
  inputs_sent = [x.lower() for x in inputs_sent]
  dialog_sent = [x.lower() for x in dialog_sent]
  inputs_sent.append("<eos>")
  dialog_sent.append("<eos>")

  if len(inputs_sent) >= MAX_SENT_LEN:
    inputs_sent = inputs_sent[:MAX_SENT_LEN]
  else:
    for _ in range(MAX_SENT_LEN - len(inputs_sent)):
      inputs_sent.append("<pad>")

  if len(dialog_sent) >= MAX_SENT_LEN:
    dialog_sent = dialog_sent[:MAX_SENT_LEN]
  else:
    for _ in range(MAX_SENT_LEN - len(dialog_sent)):
      dialog_sent.append("<pad>")

  # add parsed sentences
  inputs.append(inputs_sent)
  output_dialog.append(dialog_sent) 

inputs_tokenized = [tokenizer.encode(" ".join(sentence), add_special_tokens=True) for sentence in inputs]
output_tokenized = [tokenizer.encode(" ".join(sentence), add_special_tokens=True) for sentence in output_dialog]

100%|██████████| 10000/10000 [00:03<00:00, 3153.54it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (591 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
print(inputs[:50])
print(output_dialog[:50])
print(words[:50])

Definiamo ora la classe per caricare il dataset e il DataLoader che useremo per effettuae il training del modello.

In [ ]:
class MTDataset(torch.utils.data.Dataset):
  def __init__(self):
    # import and initialize dataset
    self.source = np.array(inputs, dtype = int)
    self.target = np.array(dialog_sent, dtype = int)

  def __getitem__(self, idx):
    # get item by index
    return self.source[idx], self.target[idx]

  def __len__(self):
    # returns length of data
    return len(self.source)

dataset = MTDataset()


In [ ]:
NUM_INSTANCES = len(dataset)

indices = list(range(NUM_INSTANCES))

train_loader = torch.utils.data.DataLoader(dataset, batch_size = BATCH_SIZE)
test_loader = torch.utils.data.DataLoader(dataset, batch_size = BATCH_SIZE)

Come visto in precedenza, definiamo una classe di Embeddings in cui includere la trasformazione in embeddings dei nostri input e il Positional Encoding dei nostri token.

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, hidden_size, position_embeddings_size):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size,
                                             hidden_size)
        self.position_embeddings = nn.Embedding(position_embeddings_size,
                                                hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout()

    def forward(self, input_ids):
        # Create position IDs for input sequence
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length, dtype=torch.long).unsqueeze(0).to(DEVICE)
        # Create token and position embeddings
       
        position_embeddings = self.position_embeddings(position_ids)
        # Combine token and position embeddings
        embeddings = token_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings